# Notebook 07: Monitoring and Observability

## Why This Matters

ML models degrade silently in production. Unlike software bugs that produce error messages, model performance decay produces wrong predictions that often go unnoticed until business metrics drop. Companies like Zillow, whose iBuying algorithm failed partially due to insufficient model monitoring, demonstrate the real-world cost of inadequate ML observability. Google, Uber, and Meta all have dedicated ML monitoring platforms that track model health continuously. In interviews, discussing monitoring is a strong signal of production ML maturity—it shows you understand that deploying a model is not the end of the job but the beginning.

## 1. Four Pillars of ML Monitoring

Production ML monitoring requires tracking four distinct layers: infrastructure health, model operational metrics, statistical performance, and business impact. Monitoring only one layer creates blind spots.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('Four Pillars of ML Monitoring', fontsize=14, fontweight='bold')

pillars = [
    {
        'title': '1. Infrastructure\nMonitoring',
        'color': '#2C3E50',
        'x': 0.3, 'y': 5.5,
        'metrics': [
            'CPU / GPU utilization',
            'Memory usage',
            'Inference latency (p50, p99)',
            'Request error rate',
            'Throughput (req/sec)',
            'Queue depth',
        ],
        'tools': 'Prometheus, Grafana, Datadog',
        'alert_when': 'Latency SLA breach, error spike',
    },
    {
        'title': '2. Prediction\nMonitoring',
        'color': '#2980B9',
        'x': 4.3, 'y': 5.5,
        'metrics': [
            'Prediction distribution',
            'Score histogram',
            'Prediction rate (calls/sec)',
            'Null/missing prediction rate',
            'Model version distribution',
            'Confidence calibration',
        ],
        'tools': 'Evidently AI, WhyLabs, Fiddler',
        'alert_when': 'Prediction distribution shifts',
    },
    {
        'title': '3. Data Drift\nMonitoring',
        'color': '#8E44AD',
        'x': 8.3, 'y': 5.5,
        'metrics': [
            'Feature distribution (KL div)',
            'Feature missing rate',
            'Covariate shift detection',
            'Label drift (if available)',
            'New category appearance',
            'Embedding drift (cosine sim)',
        ],
        'tools': 'Evidently, Nannyml, Arize',
        'alert_when': 'PSI > 0.2, KS test p < 0.05',
    },
    {
        'title': '4. Business\nMetric Monitoring',
        'color': '#27AE60',
        'x': 12.3, 'y': 5.5,
        'metrics': [
            'CTR / Conversion rate',
            'Revenue per session',
            'User engagement metrics',
            'Complaint / reversal rate',
            'A/B test cohort monitoring',
            'Long-term user retention',
        ],
        'tools': 'Business BI tools, custom dashboards',
        'alert_when': '>2% regression in KPI',
    },
]

for p in pillars:
    rect = FancyBboxPatch((p['x'], p['y']), 3.6, 3.8,
                          boxstyle="round,pad=0.15", facecolor=p['color'],
                          edgecolor='white', linewidth=2, alpha=0.9)
    ax.add_patch(rect)
    ax.text(p['x'] + 1.8, p['y'] + 3.4, p['title'], ha='center', va='center',
            fontsize=10, fontweight='bold', color='white', multialignment='center')

    for i, metric in enumerate(p['metrics']):
        ax.text(p['x'] + 0.2, p['y'] + 2.8 - i * 0.42, f'• {metric}',
                fontsize=7.8, color='white', alpha=0.9)

    # Details below
    ax.text(p['x'] + 1.8, p['y'] - 0.5, f'Tools: {p["tools"]}',
            ha='center', fontsize=7.5, color='#555', style='italic', wrap=True)
    ax.text(p['x'] + 1.8, p['y'] - 1.0, f'Alert: {p["alert_when"]}',
            ha='center', fontsize=7.5, color='#E74C3C', fontweight='bold')

# Timeline at bottom
ax.text(8, 0.8, 'Detection speed: Infra (seconds) → Prediction (minutes) → Drift (hours) → Business (days/weeks)',
        ha='center', fontsize=9, color='#333', style='italic')

plt.tight_layout()
plt.savefig('ml_monitoring_pillars.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Detecting Model Performance Degradation

The fundamental challenge of ML monitoring is that you often don't have ground truth labels in real-time. Fraud labels may come 30 days late; recommendation quality may never have explicit labels. Proxy metrics and statistical tests fill this gap.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Simulate 90 days of model predictions with a drift event on day 45
days = np.arange(90)
drift_day = 45

# Simulate daily prediction score distributions
def generate_daily_scores(day, n=1000):
    if day < drift_day:
        return np.random.beta(2, 8, n)  # baseline: mean ~0.20
    else:
        # Gradual drift: distribution shifts
        drift_strength = min(1.0, (day - drift_day) / 20.0)
        return np.random.beta(2 + drift_strength * 3, 8 - drift_strength * 2, n)

reference_scores = generate_daily_scores(0)

# Daily monitoring statistics
daily_mean = []
daily_std = []
ks_pvalues = []
psi_values = []

def compute_psi(expected, actual, n_bins=10):
    """Population Stability Index — standard drift metric in finance/ML."""
    bins = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    bins[0] = -np.inf; bins[-1] = np.inf
    
    exp_counts = np.histogram(expected, bins=bins)[0] + 0.0001
    act_counts = np.histogram(actual, bins=bins)[0] + 0.0001
    
    exp_pct = exp_counts / exp_counts.sum()
    act_pct = act_counts / act_counts.sum()
    
    psi = np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))
    return psi

for day in days:
    daily = generate_daily_scores(day)
    daily_mean.append(daily.mean())
    daily_std.append(daily.std())
    _, ks_p = stats.ks_2samp(reference_scores, daily)
    ks_pvalues.append(ks_p)
    psi_values.append(compute_psi(reference_scores, daily))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Model Monitoring Dashboard: Detecting Distribution Drift', fontsize=13, fontweight='bold')

ax = axes[0][0]
ax.plot(days, daily_mean, 'b-', linewidth=2, label='Daily mean score')
ax.fill_between(days,
                [m - s for m, s in zip(daily_mean, daily_std)],
                [m + s for m, s in zip(daily_mean, daily_std)],
                alpha=0.2, color='blue', label='±1 std dev')
ax.axvline(x=drift_day, color='red', linestyle='--', linewidth=2, label='Drift event')
ax.set_xlabel('Day'); ax.set_ylabel('Prediction Score')
ax.set_title('Prediction Score Distribution Over Time')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax2 = axes[0][1]
ax2.plot(days, ks_pvalues, 'g-', linewidth=2, label='KS test p-value')
ax2.axhline(y=0.05, color='orange', linestyle='--', label='α=0.05 threshold')
ax2.axhline(y=0.01, color='red', linestyle='--', label='α=0.01 threshold')
ax2.axvline(x=drift_day, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Day'); ax2.set_ylabel('p-value')
ax2.set_title('KS Test for Distribution Change')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

ax3 = axes[1][0]
colors = ['#E74C3C' if p > 0.2 else '#F39C12' if p > 0.1 else '#27AE60' for p in psi_values]
ax3.bar(days, psi_values, color=colors, alpha=0.7)
ax3.axhline(y=0.1, color='orange', linestyle='--', label='PSI=0.1 (monitor closely)')
ax3.axhline(y=0.2, color='red', linestyle='--', label='PSI=0.2 (significant drift)')
ax3.axvline(x=drift_day, color='red', linestyle='--', linewidth=2)
ax3.set_xlabel('Day'); ax3.set_ylabel('Population Stability Index (PSI)')
ax3.set_title('PSI Drift Detection')
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)

# Score distribution comparison
ax4 = axes[1][1]
scores_early = generate_daily_scores(10, n=5000)
scores_late = generate_daily_scores(80, n=5000)
ax4.hist(scores_early, bins=50, density=True, alpha=0.6, color='blue', label='Days 1-40 (baseline)')
ax4.hist(scores_late, bins=50, density=True, alpha=0.6, color='red', label='Days 60-90 (after drift)')
ax4.set_xlabel('Prediction Score'); ax4.set_ylabel('Density')
ax4.set_title('Score Distribution: Before vs After Drift')
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('drift_monitoring.png', dpi=120, bbox_inches='tight')
plt.show()

# When does each detector fire?
ks_detection = next((d for d, p in enumerate(ks_pvalues) if d > drift_day and p < 0.05), None)
psi_detection = next((d for d, p in enumerate(psi_values) if d > drift_day and p > 0.1), None)
mean_detection = next((d for d, m in enumerate(daily_mean) if d > drift_day and m > daily_mean[0] * 1.2), None)

print(f"Drift occurred on: Day {drift_day}")
print(f"Mean shift detected: Day {mean_detection} (+{mean_detection - drift_day if mean_detection else 'never'} days lag)")
print(f"PSI detected:       Day {psi_detection} (+{psi_detection - drift_day if psi_detection else 'never'} days lag)")
print(f"KS test detected:   Day {ks_detection} (+{ks_detection - drift_day if ks_detection else 'never'} days lag)")

## 3. Label Delay and Proxy Metrics

Ground truth labels often arrive with delay. Fraud labels come after chargebacks (30-60 days). Recommendation quality may never have explicit labels. This creates the 'monitoring gap' where model degradation isn't immediately measurable.

In [ ]:
import pandas as pd

# Label delay by use case and proxy metrics
monitoring_strategies = pd.DataFrame([
    {
        'Use Case': 'Fraud Detection',
        'Label Type': 'Chargeback / dispute',
        'Label Delay': '30-90 days',
        'Proxy Metrics': 'Rule-based flag rate, manual review rate, known fraud rate subset',
        'Statistical Monitors': 'Score distribution, PSI, feature drift',
        'Risk': 'HIGH — delayed labels mask emerging fraud patterns',
    },
    {
        'Use Case': 'Content Recommendation',
        'Label Type': 'User satisfaction (implicit)',
        'Label Delay': 'Never explicit',
        'Proxy Metrics': 'CTR, watch time, like rate, save rate, skip rate',
        'Statistical Monitors': 'Score distribution, coverage, diversity',
        'Risk': 'MEDIUM — proxy metrics can diverge from true satisfaction',
    },
    {
        'Use Case': 'Ads CTR Prediction',
        'Label Type': 'Click (observed in real-time)',
        'Label Delay': 'Minutes',
        'Proxy Metrics': 'Click rate, calibration (predicted vs actual CTR)',
        'Statistical Monitors': 'Calibration curve, AUC rolling window',
        'Risk': 'LOW — fast label feedback enables quick detection',
    },
    {
        'Use Case': 'Credit Scoring',
        'Label Type': 'Default (missed payment)',
        'Label Delay': '6-24 months',
        'Proxy Metrics': 'Score distribution, approval rate, early delinquency',
        'Statistical Monitors': 'PSI heavily used in finance, Gini coefficient',
        'Risk': 'HIGH — model may be wrong for a year before knowing',
    },
    {
        'Use Case': 'ETA Prediction (Uber/DoorDash)',
        'Label Type': 'Actual arrival time',
        'Label Delay': 'Minutes to hours',
        'Proxy Metrics': 'RMSE on completed trips, prediction bias by route type',
        'Statistical Monitors': 'Rolling RMSE, segment analysis (rush hour, rain)',
        'Risk': 'LOW-MEDIUM — labels available but slow data accumulation',
    },
])

print("Label Delay and Monitoring Strategy by Use Case")
print("=" * 80)
for _, row in monitoring_strategies.iterrows():
    print(f"\n[{row['Use Case']}]")
    print(f"  Label: {row['Label Type']} (delay: {row['Label Delay']})")
    print(f"  Proxy metrics: {row['Proxy Metrics']}")
    print(f"  Statistical: {row['Statistical Monitors']}")
    print(f"  Risk: {row['Risk']}")

print("\n" + "="*70)
print("General strategy when labels are delayed:")
print("  1. Monitor input feature distributions (data drift)")
print("  2. Monitor prediction score distributions")
print("  3. Use proxy business metrics as early warning signals")
print("  4. Maintain a labeled holdout for delayed offline evaluation")
print("  5. Use shadow models to compare against reference")

## 4. Alerting and Response Playbooks

Good monitoring without clear alerting thresholds and response procedures is incomplete. The alerting system must balance sensitivity (catch real issues) vs specificity (avoid alert fatigue). Every production ML system needs documented response playbooks.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

alert_system = {
    'CRITICAL (page on-call immediately)': {
        'color': '#E74C3C',
        'conditions': [
            'Inference error rate > 1% (requests returning errors)',
            'p99 latency > 2× SLA for 5+ minutes',
            'Prediction rate drops > 50% (model not being called)',
            'Model serving completely down',
            'Primary business metric drops > 5% (CTR, revenue)',
        ],
        'response': 'Rollback to previous model version immediately, then investigate',
    },
    'WARNING (Slack alert, fix within hours)': {
        'color': '#E67E22',
        'conditions': [
            'PSI > 0.2 for any key feature',
            'Prediction score mean shifts > 2 std devs from baseline',
            'New category/value in categorical feature not seen in training',
            'Missing feature rate > 10% for any feature',
            'Business metric drops > 2% sustained over 2 hours',
        ],
        'response': 'Investigate drift cause, consider triggering retraining',
    },
    'INFO (Daily report, fix within days)': {
        'color': '#F39C12',
        'conditions': [
            'PSI between 0.1-0.2 for any feature',
            'Prediction score distribution shift but within bounds',
            'Model version lag > 7 days since last retrain',
            'Offline-online metric gap widening',
        ],
        'response': 'Schedule retraining, document in model health log',
    },
}

for severity, info in alert_system.items():
    print(f"\n{'='*65}")
    print(f"  {severity}")
    print(f"{'='*65}")
    for condition in info['conditions']:
        print(f"  • {condition}")
    print(f"  ▶ Response: {info['response']}")

print("\n" + "="*65)
print("Response Playbook: Model Degradation")
print("="*65)
steps = [
    "1. DETECT: Alert fires on degradation signal",
    "2. TRIAGE: Is it data pipeline issue or model issue?",
    "   a. Check feature availability in online store",
    "   b. Check upstream data pipeline health",
    "   c. Check if drift is in inputs or predictions",
    "3. CONTAIN: If severe, rollback to previous model version",
    "4. DIAGNOSE: Identify root cause (data drift, concept drift, bug)",
    "5. FIX: Trigger retraining / fix pipeline / update features",
    "6. VALIDATE: Shadow test or canary before re-deploying",
    "7. DOCUMENT: Add to model health log, update monitoring rules",
]
for step in steps:
    print(f"  {step}")

## 5. Observability Stack for ML Systems

A complete ML observability stack integrates infrastructure monitoring, data quality monitoring, model performance monitoring, and business metric dashboards. Understanding which tools serve which purpose is important for system design discussions.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(15, 10))
ax.set_xlim(0, 15); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('ML Observability Stack', fontsize=14, fontweight='bold')

layers = [
    {
        'name': 'Business Layer',
        'y': 8.0,
        'color': '#27AE60',
        'components': [
            ('Business Dashboards', 'Looker / Tableau / Mode'),
            ('A/B Test Platform', 'Internal / LaunchDarkly'),
            ('Revenue Tracking', 'Custom BI'),
        ]
    },
    {
        'name': 'ML Model Layer',
        'y': 5.8,
        'color': '#8E44AD',
        'components': [
            ('Drift Detection', 'Evidently / Arize / Fiddler'),
            ('Model Performance', 'MLflow / W&B / Comet'),
            ('Feature Monitoring', 'Tecton / Feast monitors'),
        ]
    },
    {
        'name': 'Application Layer',
        'y': 3.6,
        'color': '#2980B9',
        'components': [
            ('Request Logging', 'Kafka → S3 / BigQuery'),
            ('Distributed Tracing', 'Jaeger / Zipkin / DataDog APM'),
            ('Error Tracking', 'Sentry / Rollbar'),
        ]
    },
    {
        'name': 'Infrastructure Layer',
        'y': 1.4,
        'color': '#E74C3C',
        'components': [
            ('Metrics Collection', 'Prometheus'),
            ('Visualization', 'Grafana'),
            ('Alerting', 'PagerDuty / OpsGenie'),
        ]
    },
]

for layer in layers:
    # Layer header
    ax.add_patch(FancyBboxPatch((0.2, layer['y']), 14.6, 1.8,
                                boxstyle="round,pad=0.1", facecolor=layer['color'],
                                edgecolor='white', linewidth=2, alpha=0.12))
    ax.text(0.5, layer['y'] + 1.5, layer['name'], fontsize=10, fontweight='bold',
            color=layer['color'])

    for j, (comp_name, tools) in enumerate(layer['components']):
        x = 1 + j * 4.8
        ax.add_patch(FancyBboxPatch((x, layer['y'] + 0.1), 4.2, 1.2,
                                    boxstyle="round,pad=0.1", facecolor=layer['color'],
                                    edgecolor='white', linewidth=1.5, alpha=0.85))
        ax.text(x + 2.1, layer['y'] + 0.85, comp_name, ha='center', va='center',
                fontsize=8.5, fontweight='bold', color='white')
        ax.text(x + 2.1, layer['y'] + 0.38, tools, ha='center', va='center',
                fontsize=7.5, color='white', alpha=0.9)

# Arrows between layers
for y in [3.4, 5.6, 7.8]:
    ax.annotate('', xy=(7.5, y + 0.2), xytext=(7.5, y),
                arrowprops=dict(arrowstyle='->', color='#aaa', lw=2))

ax.text(7.5, 0.5, 'All layers feed into: On-call Alerting → Incident Response → Post-mortem',
        ha='center', fontsize=9, color='#666', style='italic')

plt.tight_layout()
plt.savefig('observability_stack.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

| Monitoring Type | What to Track | Key Metrics | Alert Threshold |
|-----------------|---------------|-------------|------------------|
| Infrastructure | Latency, errors, throughput | p99 latency, error rate, QPS | Latency 2× SLA; error > 1% |
| Prediction | Score distributions | Mean, std, histogram | PSI > 0.2, mean shift > 2σ |
| Data drift | Feature distributions | PSI, KS statistic, KL divergence | PSI > 0.1 warning, > 0.2 critical |
| Business metrics | Downstream KPIs | CTR, revenue, engagement | > 2-5% regression |

**Key Principles**
- Monitor inputs AND outputs — data drift often causes model degradation without triggering errors
- Label delay is real — use proxy metrics and statistical monitors when labels are slow
- PSI (Population Stability Index) is the standard drift metric in production ML (from finance)
- Alert thresholds need tuning — too sensitive creates alert fatigue, too lenient misses problems
- Monitoring is a feedback loop to trigger retraining, not just a pager alert system
- Shadow models provide a reference baseline to detect regressions